In [ ]:
!pip install Bio
!pip install torch_geometric
!pip install gprofiler-official 

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_selection import SelectFromModel
from tqdm.auto import tqdm
import lightgbm as lgb
import gc
import plotly.graph_objects as go
import plotly.express as px
import networkx as nx
import re
import torch_geometric.transforms as T
from torch_geometric.data import Data
from torch_geometric.nn import GCNConv, VGAE
from torch_geometric.utils import negative_sampling

warnings.filterwarnings('ignore')

# --- Configuration ---
TAXONOMIC_PROFILE_FILE = "/kaggle/input/microbiome-cytokine/taxonomic_profiles.csv" 
CYTOKINE_FILE = "/kaggle/input/microbiome-cytokine/cytokine_profiles.csv"
MAPPING_FILE = "/kaggle/input/microbiome-cytokine/Train (5).csv"

# --- Helper Functions ---
def clean_feature_names(feature_names):
    """Clean feature names by replacing special characters with underscores"""
    cleaned_names = []
    for name in feature_names:
        cleaned = re.sub(r'[^a-zA-Z0-9_]', '_', str(name))
        cleaned = re.sub(r'_+', '_', cleaned).strip('_')
        if not cleaned:
            cleaned = 'feature_' + str(hash(name))[:8]
        cleaned_names.append(cleaned)
    return cleaned_names

# --- VGAE Model Definition ---
class VariationalGCNEncoder(torch.nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv1 = GCNConv(in_channels, 2 * out_channels)
        self.conv_mu = GCNConv(2 * out_channels, out_channels)
        self.conv_logstd = GCNConv(2 * out_channels, out_channels)

    def forward(self, x, edge_index):
        x = F.relu(self.conv1(x, edge_index))
        return self.conv_mu(x, edge_index), self.conv_logstd(x, edge_index)

# --- Main Execution ---
if __name__ == '__main__':
    print(" Starting Body-Site-Aware Network Discovery with a Variational Graph Autoencoder (VGAE)")

    # 1. Load and Merge Data
    print("\n--- Step 1: Loading and Preparing Datasets ---")
    abundance_df = pd.read_csv(TAXONOMIC_PROFILE_FILE)
    cytokine_df = pd.read_csv(CYTOKINE_FILE)
    mapping_df = pd.read_csv(MAPPING_FILE)
    
    # Standardize the filename format for merging using filename_stem
    abundance_df.rename(columns={'SampleID': 'filename_stem'}, inplace=True)
    mapping_df['filename_stem'] = mapping_df['filename'].str.replace('.mgb', '', regex=False)
    
    # Perform the merge on filename_stem
    master_df = pd.merge(
        mapping_df[['SampleID', 'SampleType', 'filename_stem', 'SubjectID']],
        abundance_df,
        on='filename_stem',
        how='inner'
    )
    
    # Merge with cytokine data using SampleID
    master_df = pd.merge(master_df, cytokine_df, on='SampleID', how='inner')
    
    print(f"✅ Data prepared. Full dataset has {master_df.shape[0]} samples.")

    # Define feature and target columns
    microbe_cols = [col for col in abundance_df.columns if col not in ['filename_stem']]
    non_cytokine_cols = ['SampleID', 'Plate', 'CollectionDate', 'CL1', 'CL2', 'CL3', 'CL4', 'filename_stem', 'SubjectID']
    cytokine_cols = [col for col in cytokine_df.columns if col not in non_cytokine_cols]
    
    # Clean microbe column names for compatibility
    original_microbe_cols = microbe_cols.copy()
    cleaned_microbe_cols = clean_feature_names(microbe_cols)
    microbe_col_mapping = dict(zip(original_microbe_cols, cleaned_microbe_cols))
    
    # Create reverse mapping for results
    reverse_microbe_mapping = {v: k for k, v in microbe_col_mapping.items()}
    
    body_sites = master_df['SampleType'].unique()
    print(f"🔬 Found {len(body_sites)} unique body sites: {body_sites}")
    
    for site in body_sites:
        print(f"\n{'='*60}\nProcessing Body Site: {site.upper()}\n{'='*60}")
        
        site_df = master_df[master_df['SampleType'] == site].copy()
        if len(site_df) < 30:
            print(f"    -> Skipping site '{site}' due to insufficient samples.")
            continue
            
        print(f"    -> Building network from {len(site_df)} samples...")
        
        # 2. Create BODY-SITE-SPECIFIC node mapping
        print("    -> Creating body-site-specific node mapping...")
        
        # Identify which microbes and cytokines are actually present in this site
        present_microbes = []
        present_cytokines = []
        
        # Check which microbes have non-zero presence in this site
        for microbe in original_microbe_cols:
            if site_df[microbe].sum() > 0:
                present_microbes.append(microbe)
        
        # Check which cytokines have non-missing values in this site
        for cytokine in cytokine_cols:
            if not site_df[cytokine].isna().all():
                present_cytokines.append(cytokine)
        
        print(f"    -> Site has {len(present_microbes)} microbes and {len(present_cytokines)} cytokines")
        
        # Create local node mapping for this specific body site
        local_nodes = []
        node_name_to_idx = {}
        
        # Add present microbes with cleaned names
        for i, microbe in enumerate(present_microbes):
            cleaned_name = microbe_col_mapping[microbe]
            local_nodes.append({'original_name': microbe, 'cleaned_name': cleaned_name, 'type': 'microbe'})
            node_name_to_idx[cleaned_name] = i
        
        # Add present cytokines
        for j, cytokine in enumerate(present_cytokines):
            idx = len(present_microbes) + j
            local_nodes.append({'original_name': cytokine, 'cleaned_name': cytokine, 'type': 'cytokine'})
            node_name_to_idx[cytokine] = idx
        
        total_nodes = len(present_microbes) + len(present_cytokines)
        print(f"    -> Total nodes in graph: {total_nodes}")
        
        # Create node features (mean abundance/expression for each node)
        node_features = []
        for node_info in local_nodes:
            if node_info['type'] == 'microbe':
                # Use mean abundance of the microbe across samples
                feature_value = site_df[node_info['original_name']].mean()
            else:
                # Use mean cytokine level across samples
                feature_value = site_df[node_info['original_name']].mean()
            node_features.append(feature_value)
        
        node_features = torch.tensor(node_features, dtype=torch.float).reshape(-1, 1)
        scaler = StandardScaler()
        scaled_features = scaler.fit_transform(node_features.numpy())
        node_features = torch.tensor(scaled_features, dtype=torch.float)
        
        # 3. Use RANDOM FOREST Feature Importance to Define Edges
        edge_list = []
        print("    -> Using Random Forest for feature importance...")
        
        for cytokine in tqdm(present_cytokines, desc=f"Finding RF links for {site}", leave=False):
            # Prepare data for Random Forest
            X_rf = site_df[present_microbes].copy()
            X_rf.columns = [microbe_col_mapping[m] for m in present_microbes]
            
            y_rf = site_df[cytokine].dropna()
            X_rf = X_rf.loc[y_rf.index]
            
            # Check for sufficient data
            if len(y_rf) < 10 or y_rf.nunique() <= 1:
                continue
                
            # Use Random Forest with feature selection
            rf_model = RandomForestRegressor(
                n_estimators=100,
                max_depth=5,
                random_state=42,
                n_jobs=-1
            )
            
            try:
                rf_model.fit(X_rf, y_rf)
                
                # Get feature importances
                importances = pd.Series(rf_model.feature_importances_, index=X_rf.columns)
                
                # Select top features with importance > 0
                top_features = importances[importances > 0].nlargest(10)
                
                for feature, importance in top_features.items():
                    u = node_name_to_idx[feature]
                    v = node_name_to_idx[cytokine]
                    
                    # Verify node indices are valid
                    if u < total_nodes and v < total_nodes:
                        edge_list.append((u, v))
                        edge_list.append((v, u))  # Add reverse edge for undirected graph
                        
            except Exception as e:
                continue
                    
        if not edge_list:
            print(f"    -> WARNING: No important feature links found for site '{site}'. Skipping.")
            continue
        
        # Remove duplicate edges and ensure they're within valid range
        unique_edges = list(set(edge_list))
        unique_edges = [(u, v) for u, v in unique_edges if u < total_nodes and v < total_nodes]
        
        if not unique_edges:
            print(f"    -> ERROR: No valid edges found after filtering. Skipping site.")
            continue
        
        edge_index = torch.tensor(unique_edges, dtype=torch.long).t().contiguous()
        print(f"    -> Found {len(unique_edges)} strong links using Random Forest.")
        
        # Verify that all node indices in edges are valid
        max_node_idx = edge_index.max().item() if len(unique_edges) > 0 else -1
        if max_node_idx >= total_nodes:
            print(f"ERROR: Found node index {max_node_idx} but only {total_nodes} nodes exist!")
            continue
        
        # Create graph data with LOCAL node indices
        graph_data = Data(x=node_features, edge_index=edge_index)
        print(f"    -> Graph data created with {graph_data.num_nodes} nodes and {graph_data.num_edges} edges")

        # 4. Initialize and Train the VGAE Model
        print("    -> Training VGAE model...")
        out_channels = 32
        model = VGAE(VariationalGCNEncoder(node_features.shape[1], out_channels))
        optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
        
        # Split links for training and testing
        transform = T.RandomLinkSplit(
            num_val=0.1,
            num_test=0.1,
            is_undirected=True,
            add_negative_train_samples=True,
            split_labels=True
        )
        train_data, val_data, test_data = transform(graph_data)

        best_loss = float('inf')
        patience, patience_counter = 30, 0

        for epoch in range(1, 301):
            model.train()
            optimizer.zero_grad()
            z = model.encode(train_data.x, train_data.edge_index)
            
            # Handle different PyTorch Geometric versions
            if hasattr(train_data, 'edge_label_index') and hasattr(train_data, 'edge_label'):
                # Newer version
                link_scores = model.decode(z, train_data.edge_label_index)
                link_labels = train_data.edge_label
            else:
                # Older version - manually create negative samples
                pos_edges = train_data.edge_index
                neg_edges = negative_sampling(
                    edge_index=train_data.edge_index,
                    num_nodes=train_data.num_nodes,
                    num_neg_samples=pos_edges.size(1)
                )
                all_edges = torch.cat([pos_edges, neg_edges], dim=1)
                link_labels = torch.cat([torch.ones(pos_edges.size(1)), 
                                       torch.zeros(neg_edges.size(1))])
                link_scores = model.decode(z, all_edges)
            
            # Calculate BCE loss and add the KL divergence loss for VGAE
            recon_loss = F.binary_cross_entropy_with_logits(link_scores, link_labels)
            kl_loss = (1 / train_data.num_nodes) * model.kl_loss()
            loss = recon_loss + kl_loss
            
            loss.backward()
            optimizer.step()
            
            if epoch % 50 == 0:
                print(f"        Epoch: {epoch:03d}, Training Loss: {loss:.4f}")
            
            # Early stopping
            if loss < best_loss:
                best_loss = loss
                patience_counter = 0
                torch.save(model.state_dict(), f'best_model_{site}.pth')
            else:
                patience_counter += 1
            if patience_counter >= patience:
                print(f"        Early stopping at epoch {epoch}.")
                break

        # 5. Generate and Save Final Network
        model.load_state_dict(torch.load(f'best_model_{site}.pth'))
        print("    -> Generating final network and saving results...")
        with torch.no_grad():
            model.eval()
            z = model.encode(graph_data.x, graph_data.edge_index)
        
        # Calculate interaction scores using cosine similarity
        z_norm = F.normalize(z, p=2, dim=1)
        interaction_matrix = torch.mm(z_norm, z_norm.t()).cpu().numpy()
        
        results = []
        for i, node_info in enumerate(local_nodes):
            if node_info['type'] == 'microbe':
                for j, cytokine in enumerate(present_cytokines):
                    cytokine_idx = node_name_to_idx[cytokine]
                    score = interaction_matrix[i, cytokine_idx]
                    
                    results.append({
                        'Microbe': node_info['original_name'],
                        'Cytokine': cytokine,
                        'InteractionScore': score,
                        'BodySite': site
                    })
        
        output_df = pd.DataFrame(results).sort_values(by='InteractionScore', ascending=False)
        output_filename = f"Track3_Network_{site}_VGAE_RF_Final.csv"
        output_df.to_csv(output_filename, index=False)
        print(f"🎉 Success! Network for '{site}' saved to '{output_filename}'")
        
        # Print the top 20 interactions for the current site
        print(f"\n--- Top 20 Most Important Interactions for {site} ---")
        print(output_df.head(20).to_string(index=False))
        
        # 6. AESTHETIC NETWORK VISUALIZATION
        print("\n    -> Creating simplified aesthetic network visualization...")
        top_n_edges = min(25, len(output_df))
        site_top_df = output_df.head(top_n_edges)
        
        # Create network graph
        G = nx.from_pandas_edgelist(site_top_df, 'Microbe', 'Cytokine', ['InteractionScore'])
        
        # Create visualization directly here (no function)
        pos = nx.spring_layout(G, k=1.5, iterations=100, seed=42)
        
        # Create edge traces
        edge_x, edge_y = [], []
        for edge in G.edges():
            x0, y0 = pos[edge[0]]
            x1, y1 = pos[edge[1]]
            edge_x.extend([x0, x1, None])
            edge_y.extend([y0, y1, None])

        edge_trace = go.Scatter(
            x=edge_x, y=edge_y,
            line=dict(width=2, color='rgba(150,150,150,0.5)'),
            hoverinfo='none',
            mode='lines',
            showlegend=False
        )

        # Create node traces
        node_x, node_y, node_text, node_color, node_size = [], [], [], [], []
        for node in G.nodes():
            x, y = pos[node]
            node_x.append(x)
            node_y.append(y)
            
            # Truncate long names for better display
            display_name = node if len(node) < 20 else node[:17] + '...'
            node_text.append(f"{display_name}<br>Connections: {G.degree[node]}")
            
            # Color by node type
            if any(cytokine in node for cytokine in output_df['Cytokine'].unique()):
                node_color.append('#3498db')  # Nice blue for cytokines
                node_size.append(28)
            else:
                node_color.append('#e67e22')  # Nice orange for microbes
                node_size.append(22)

        node_trace = go.Scatter(
            x=node_x, y=node_y,
            text=node_text,
            mode='markers+text',
            textposition='middle center',
            hoverinfo='text',
            marker=dict(
                color=node_color,
                size=node_size,
                line=dict(width=2, color='white'),
                opacity=0.9
            ),
            textfont=dict(
                size=9,
                color='white',
                family="Arial"
            ),
            showlegend=False
        )

        # Create the figure
        fig = go.Figure(
            data=[edge_trace, node_trace],
            layout=go.Layout(
                title=dict(
                    text=f'<b>{site} Microbe-Cytokine Network</b><br>Top {top_n_edges} Interactions',
                    font=dict(size=20, family="Arial"),
                    x=0.5,
                    y=0.95
                ),
                showlegend=True,
                width=900,
                height=700,
                margin=dict(b=50, l=50, r=50, t=100),
                xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
                yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
                #plot_bgcolor='rgba(240,240,240,0.9)',
                plot_bgcolor='rgba(30,40,60,0.95)',  # Dark blue background
                paper_bgcolor='white',
                annotations=[
                    dict(
                        x=0.5,
                        y=-0.05,
                        xref="paper",
                        yref="paper",
                        text="<span style='color:#e67e22'>🟠 Microbes</span> | <span style='color:#3498db'>🔵 Cytokines</span>",
                        showarrow=False,
                        font=dict(size=12, family="Arial")
                    )
                ]
            )
        )

        # Add legend
        fig.add_trace(go.Scatter(
            x=[None], y=[None],
            mode='markers',
            marker=dict(size=15, color='#e67e22'),
            name='Microbes',
            showlegend=True
        ))
        
        fig.add_trace(go.Scatter(
            x=[None], y=[None],
            mode='markers',
            marker=dict(size=15, color='#3498db'),
            name='Cytokines',
            showlegend=True
        ))

        # Save and show
        viz_filename = f"Network_{site}_Simple.html"
        fig.write_html(viz_filename)
        print(f"    ->  Network visualization saved to '{viz_filename}'")
        fig.show()

        # 7. SIMPLE HEATMAP VISUALIZATION
        print("    -> Creating interaction heatmap...")
        top_20_interactions = output_df.head(20)
        
        # Create pivot table for heatmap
        heatmap_data = top_20_interactions.pivot_table(
            values='InteractionScore',
            index='Microbe',
            columns='Cytokine',
            fill_value=0
        )
        
        # Create heatmap
        heatmap_fig = px.imshow(
            heatmap_data,
            title=f'Top 20 Microbe-Cytokine Interactions - {site}',
            color_continuous_scale='Viridis',
            aspect='auto',
            labels={'color': 'Interaction Strength'}
        )
        
        heatmap_fig.update_layout(
            height=500,
            width=700,
            xaxis_title='Cytokines',
            yaxis_title='Microbes'
        )
        
        heatmap_filename = f"Heatmap_{site}.html"
        heatmap_fig.write_html(heatmap_filename)
        print(f"    -> Heatmap saved to '{heatmap_filename}'")
        heatmap_fig.show()

    print(f"\n{'='*60}\n✅ All body sites processed successfully with beautiful visualizations!")
    gc.collect()

In [ ]:
import requests
import pandas as pd
from gprofiler import GProfiler
import time
from collections import defaultdict
import json

# --- 1. MICROBIAL GENE RETRIEVAL FROM UNIPROT ---

class MicrobialPathwayAnalyzer:
    def __init__(self):
        self.gp = GProfiler(return_dataframe=True)
        self.base_url = "https://rest.uniprot.org/uniprotkb/search"
        self.microbe_gene_cache = {}
        
    def get_genes_for_microbe(self, microbe, max_results=200):
        """Retrieve gene names from UniProt for a given microbial species."""
        
        # Check cache first
        if microbe in self.microbe_gene_cache:
            return self.microbe_gene_cache[microbe]
            
        print(f"\n🔍 Searching UniProt for: {microbe}")
        
        # Try multiple query strategies for better results
        queries = [
            f'organism_name:"{microbe}" AND reviewed:true',
            f'organism_name:"{microbe}"',
            f'organism_name:"{microbe.split()[0]}" AND reviewed:true'  # Genus only
        ]
        
        all_genes = set()
        
        for query in queries:
            params = {
                "query": query,
                "fields": "accession,gene_names,organism_name,protein_name",
                "format": "json", 
                "size": max_results
            }
            
            try:
                response = requests.get(self.base_url, params=params, timeout=30)
                if response.status_code != 200:
                    continue
                    
                results = response.json().get("results", [])
                
                for entry in results:
                    # Extract gene names
                    gene_info = entry.get("genes", [])
                    for gene in gene_info:
                        if "geneName" in gene:
                            all_genes.add(gene["geneName"]["value"])
                        # Also get synonyms
                        if "synonyms" in gene:
                            for syn in gene["synonyms"]:
                                all_genes.add(syn["value"])
                
                if len(all_genes) > 50:  # If we found enough genes, break
                    break
                    
            except Exception as e:
                print(f"⚠️ Error querying UniProt for {microbe}: {e}")
                continue
            
            time.sleep(0.5)  # Rate limiting
        
        genes_list = list(all_genes)[:max_results]
        self.microbe_gene_cache[microbe] = genes_list
        
        print(f"✅ Found {len(genes_list)} genes for {microbe}")
        return genes_list
    
    def get_kegg_pathways_for_genes(self, genes, organism_code='ko'):
        """Use KEGG API to get pathways for gene list."""
        
        if not genes:
            return []
            
        print(f"🔬 Querying KEGG for {len(genes)} genes...")
        
        # KEGG REST API for pathway mapping
        kegg_base = "https://rest.kegg.jp"
        pathways = set()
        
        # Process genes in batches
        batch_size = 10
        for i in range(0, len(genes), batch_size):
            batch = genes[i:i+batch_size]
            gene_query = "+".join(batch)
            
            try:
                # Find KEGG gene IDs
                find_url = f"{kegg_base}/find/genes/{gene_query}"
                response = requests.get(find_url, timeout=10)
                
                if response.status_code == 200:
                    lines = response.text.strip().split('\n')
                    kegg_genes = []
                    
                    for line in lines:
                        if '\t' in line:
                            kegg_id = line.split('\t')[0]
                            kegg_genes.append(kegg_id)
                    
                    # Get pathways for these KEGG genes
                    for kegg_gene in kegg_genes[:5]:  # Limit to avoid timeout
                        try:
                            pathway_url = f"{kegg_base}/link/pathway/{kegg_gene}"
                            path_response = requests.get(pathway_url, timeout=5)
                            
                            if path_response.status_code == 200:
                                path_lines = path_response.text.strip().split('\n')
                                for path_line in path_lines:
                                    if '\t' in path_line:
                                        pathway = path_line.split('\t')[1]
                                        pathways.add(pathway)
                        except:
                            continue
                            
            except Exception as e:
                print(f"⚠️ KEGG query error: {e}")
                continue
            
            time.sleep(0.2)  # Rate limiting
        
        return list(pathways)
    
    def analyze_microbe_pathways(self, microbes, site_name, method='uniprot'):
        """Main function to analyze microbial pathways."""
        
        print(f"\n🧬 Analyzing microbial pathways for {site_name} using {method} method")
        print("="*60)
        
        all_genes = []
        microbe_results = {}
        
        for microbe in microbes:
            genes = self.get_genes_for_microbe(microbe)
            microbe_results[microbe] = genes
            all_genes.extend(genes)
        
        # Remove duplicates while preserving order
        unique_genes = list(dict.fromkeys(all_genes))
        
        print(f"\n📊 Summary for {site_name}:")
        print(f"  • Total microbes analyzed: {len(microbes)}")
        print(f"  • Total unique genes found: {len(unique_genes)}")
        
        if not unique_genes:
            print("❌ No genes found for pathway analysis")
            return None, microbe_results
        
        # Method 1: Try GProfiler with microbial genes as human orthologs
        gprofiler_results = None
        try:
            print(f"\n🎯 Running GProfiler pathway analysis...")
            
            # Limit genes to avoid timeout
            gene_subset = unique_genes[:500] if len(unique_genes) > 500 else unique_genes
            
            gprofiler_results = self.gp.profile(
                organism='hsapiens',  # Use human as reference
                query=gene_subset,
                sources=['KEGG', 'GO:BP', 'REAC'],
                user_threshold=0.1,  # More lenient threshold
                no_evidences=False
            )
            
            if not gprofiler_results.empty:
                print("✅ GProfiler analysis successful!")
                print(f"Found {len(gprofiler_results)} enriched pathways")
                print("\nTop enriched pathways:")
                top_results = gprofiler_results.head()
                for _, row in top_results.iterrows():
                    print(f"  • {row['source']}: {row['description']} (p={row['p_value']:.2e})")
            else:
                print("⚠️ No significant pathways found in GProfiler")
                
        except Exception as e:
            print(f"❌ GProfiler analysis failed: {e}")
        
        # Method 2: KEGG pathway mapping
        kegg_pathways = []
        try:
            print(f"\n🗺️ Running KEGG pathway mapping...")
            kegg_pathways = self.get_kegg_pathways_for_genes(unique_genes[:100])  # Limit for speed
            
            if kegg_pathways:
                print(f"✅ Found {len(kegg_pathways)} KEGG pathways")
                print("KEGG pathways identified:")
                for pathway in kegg_pathways[:10]:  # Show top 10
                    print(f"  • {pathway}")
            else:
                print("⚠️ No KEGG pathways found")
                
        except Exception as e:
            print(f"❌ KEGG analysis failed: {e}")
        
        return {
            'gprofiler_results': gprofiler_results,
            'kegg_pathways': kegg_pathways,
            'total_genes': len(unique_genes),
            'microbe_gene_counts': {m: len(genes) for m, genes in microbe_results.items()}
        }, microbe_results

# --- 2. INTEGRATION WITH EXISTING DATA ---

# original data
all_data = {
    'Stool': {
        'microbes': [
            'Alistipes communis', 'Megasphaera massiliensis', 'Clostridium sp. M62/1',
            'Filifactor alocis', 'Alistipes senegalensis', 'Coprococcus eutactus',
            'Segatella copri', 'Megamonas funiformis', 'Sellimonas intestinalis',
            'Enterocloster asparagiformis', 'Barnesiella viscericola'
        ],
        'cytokines': [
            'CD40L', 'NGF', 'CHEX1', 'EGF', 'PAI1', 'FGFB', 'IL18', 
            'IL15', 'IL12P70', 'IL8', 'TGFA', 'RESISTIN', 'IL4', 'TRAIL', 'VCAM1'
        ]
    },
    'Nasal': {
        'microbes': [
            'Priestia aryabhattai', 'Staphylococcus warneri', 
            'Moraxella catarrhalis', 'Methylobacterium tardum', 'Kocuria palustris', 
            'Paenibacillus swuensis', 'Campylobacter ureolyticus', 'Finegoldia magna',
            'Staphylococcus capitis'
        ],
        'cytokines': [
            'IL23', 'IFNA', 'IL27', 'GROA', 'FGFB', 'MCSF', 'FASL', 'IL7',
            'VCAM1', 'SDF1A', 'TGFB', 'IL1A', 'PAI1', 'LIF', 'NGF'
        ]
    },
    'Mouth': {
        'microbes': [
            'Porphyromonas sp. oral taxon 275', 'Prevotella sp. oral taxon 299',
            'Rothia mucilaginosa', 'Proteus mirabilis', 'Tannerella serpentiformis',
            'Prevotella histicola', 'Capnocytophaga sp. oral taxon 878'
        ],
        'cytokines': [
            'FASL', 'IL15', 'IL6', 'TNFA', 'CD40L', 'LIF', 'IL9', 'IFNA',
            'IL8', 'VCAM1', 'MCSF', 'IFNG', 'IL21', 'IL27', 'IFNB'
        ]
    },
    'Skin': {
        'microbes': [
            'Brevibacterium sp. CS2', 'Campylobacter hominis', 'Roseburia hominis',
            'Cutibacterium avidum', 'Dermabacter jinjuensis', 'Streptococcus iniae'
        ],
        'cytokines': [
            'FGFB', 'IL10', 'MCP3', 'VEGFD', 'MIP1B', 'MCSF', 'TGFB',
            'VEGF', 'IL9', 'NGF', 'IL4', 'TNFB', 'IL1B', 'SDF1A'
        ]
    }
}

# --- 3. COMPLETE ANALYSIS PIPELINE ---

def run_complete_analysis():
    """Run the complete cytokine + microbe pathway analysis."""
    
    analyzer = MicrobialPathwayAnalyzer()
    gp = GProfiler(return_dataframe=True)
    
    all_results = {}
    
    for site, data in all_data.items():
        print(f"\n{'='*80}")
        print(f"🔬 COMPLETE ANALYSIS FOR {site.upper()}")
        print('='*80)
        
        site_results = {}
        
        # --- CYTOKINE ANALYSIS ---
        print(f"\n🧪 CYTOKINE PATHWAY ANALYSIS")
        print("-" * 40)
        
        try:
            cytokine_results = gp.profile(
                organism='hsapiens',
                query=data['cytokines'],
                sources=['KEGG', 'REAC', 'GO:BP'],
                user_threshold=0.05
            )
            
            if not cytokine_results.empty:
                print(f"✅ Found {len(cytokine_results)} enriched cytokine pathways")
                print("\nTop 5 cytokine pathways:")
                for i, (_, row) in enumerate(cytokine_results.head().iterrows()):
                    print(f"  {i+1}. {row['source']}: {row['description'][:60]}... (p={row['p_value']:.2e})")
                site_results['cytokines'] = cytokine_results
            else:
                print("⚠️ No significant cytokine pathways found")
                site_results['cytokines'] = None
                
        except Exception as e:
            print(f"❌ Cytokine analysis failed: {e}")
            site_results['cytokines'] = None
        
        # --- MICROBE ANALYSIS ---
        print(f"\n🦠 MICROBIAL PATHWAY ANALYSIS")
        print("-" * 40)
        
        microbe_analysis, microbe_genes = analyzer.analyze_microbe_pathways(
            data['microbes'], 
            site,
            method='uniprot'
        )
        
        site_results['microbes'] = microbe_analysis
        site_results['microbe_genes'] = microbe_genes
        
        # --- SUMMARY ---
        print(f"\n📋 SUMMARY FOR {site}")
        print("-" * 40)
        
        cytokine_count = len(cytokine_results) if 'cytokine_results' in locals() and not cytokine_results.empty else 0
        microbe_pathway_count = 0
        
        if microbe_analysis and microbe_analysis.get('gprofiler_results') is not None:
            microbe_pathway_count = len(microbe_analysis['gprofiler_results'])
        
        print(f"  • Cytokines analyzed: {len(data['cytokines'])}")
        print(f"  • Cytokine pathways found: {cytokine_count}")
        print(f"  • Microbes analyzed: {len(data['microbes'])}")
        
        if microbe_analysis:
            print(f"  • Microbial genes retrieved: {microbe_analysis.get('total_genes', 0)}")
            print(f"  • Microbial pathways found: {microbe_pathway_count}")
            
            kegg_count = len(microbe_analysis.get('kegg_pathways', []))
            if kegg_count > 0:
                print(f"  • KEGG pathways found: {kegg_count}")
        
        all_results[site] = site_results
        
        print(f"\n" + "="*80 + "\n")
    
    # --- CROSS-SITE COMPARISON ---
    print(f"\n🌐 CROSS-SITE PATHWAY COMPARISON")
    print("="*60)
    
    for site, results in all_results.items():
        cytokine_paths = results.get('cytokines')
        microbe_analysis = results.get('microbes')
        
        print(f"\n{site}:")
        if cytokine_paths is not None and not cytokine_paths.empty:
            top_cytokine = cytokine_paths.iloc[0]['description'][:50]
            print(f"  🧪 Top cytokine pathway: {top_cytokine}...")
        
        if microbe_analysis and microbe_analysis.get('gprofiler_results') is not None:
            gprofiler_res = microbe_analysis['gprofiler_results']
            if not gprofiler_res.empty:
                top_microbe = gprofiler_res.iloc[0]['description'][:50]
                print(f"  🦠 Top microbe pathway: {top_microbe}...")
    
    return all_results

# --- 4. SAVE RESULTS FUNCTION ---

def save_results_to_files(results):
    """Save analysis results to CSV files."""
    
    print(f"\n💾 SAVING RESULTS TO FILES")
    print("-" * 40)
    
    for site, site_data in results.items():
        # Save cytokine results
        if site_data.get('cytokines') is not None:
            cytokine_file = f"{site.lower()}_cytokine_pathways.csv"
            site_data['cytokines'].to_csv(cytokine_file, index=False)
            print(f"✅ Saved cytokine pathways: {cytokine_file}")
        
        # Save microbe results
        if site_data.get('microbes') and site_data['microbes'].get('gprofiler_results') is not None:
            microbe_file = f"{site.lower()}_microbe_pathways.csv"
            site_data['microbes']['gprofiler_results'].to_csv(microbe_file, index=False)
            print(f"✅ Saved microbe pathways: {microbe_file}")
        
        # Save gene mappings
        if site_data.get('microbe_genes'):
            gene_mapping = []
            for microbe, genes in site_data['microbe_genes'].items():
                for gene in genes:
                    gene_mapping.append({'Microbe': microbe, 'Gene': gene, 'Site': site})
            
            if gene_mapping:
                gene_df = pd.DataFrame(gene_mapping)
                gene_file = f"{site.lower()}_microbe_genes.csv"
                gene_df.to_csv(gene_file, index=False)
                print(f"✅ Saved gene mappings: {gene_file}")

# --- 5. RUN THE ANALYSIS ---

if __name__ == "__main__":
    print("🚀 Starting Complete Microbiome-Cytokine Pathway Analysis")
    print("="*80)
    
    # Run the complete analysis
    results = run_complete_analysis()
    
    # Save results
    save_results_to_files(results)
    
    print(f"\n🎉 ANALYSIS COMPLETE!")
    print("="*80)
    print("Files generated:")
    print("  • [site]_cytokine_pathways.csv - Cytokine pathway enrichment results")
    print("  • [site]_microbe_pathways.csv - Microbial pathway enrichment results") 
    print("  • [site]_microbe_genes.csv - Microbe-to-gene mappings")
    print("\nNext steps:")
    print("  1. Review the pathway results for biological insights")
    print("  2. Cross-reference with literature for validation")
    print("  3. Consider functional validation experiments")
